# Numerical Analysis Project 1
##  Authors: Bopeng Zhang (bz292), Daniel Chuang (dc863), Tianrui Wang (tw559)

In [1]:
# Project Setup
import numpy as np
from typing import *

## Goal 1:
Implement the Householder scheme for computing QR factorizations. Your QR factorization routine should take in a $n \times k$ matrix $B$ with $k\leq n$ and returns a $n\times k$ matrix $Q$ with orthonormal columns (or sufficient information to be able to apply it to a vector, e.g., the Householder vectors), and a $k\times k$ upper triangular matrix $R.$ Please demonstrate that your algorithm behaves as expected (this means both that you are getting a valid QR factorization as output and that it achieves the desired computational scaling). Demonstrate that your implementation achieves the expected scaling and explain how you tested your implementation.

In [ ]:
# Translated from David Bindel's CS 4220 Notes
# Source: https://www.cs.cornell.edu/courses/cs4220/2023sp/lec/2023-02-22.html

def QR_Householder_helper(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    m, n = A.shape
    A = A.copy() # 
    tau = np.zeros(n)

    for j in range(n):
        normx = np.linalg.norm(A[j:, j])
        if normx < 1e-14: # Avoid div by 0 if normx very small
            continue
        s = -np.sign(A[j, j])
        s = -1 if A[j, j] == 0 else s
        u1 = A[j, j] - s * normx
        w = A[j:, j] / u1
        w[0] = 1.0
        if j + 1 < m:
            A[j+1:, j] = w[1:]
        A[j, j] = s * normx
        tau[j] = -s * u1 / normx
        if j + 1 < n: # Ensure we have columns to update
            # First calculate w' * A[j:, j+1:]
            wTA = w @ A[j:, j+1:] # This is a row vector (1 x n-j)
            # Then calculate w * (w'*A[j:, j+1:])
            # w is a column vector, wTA is a row vector
            # Outer product to get a matrix of the right shape
            update = np.outer(w, wTA)
            # Apply the update
            A[j:, j+1:] -= tau[j] * update
    return A, tau

def form_Q(A_mod: np.ndarray, tau: np.ndarray) -> np.ndarray:
    """
    Form the Q matrix from the Householder reflectors stored in A_mod and tau.
    
    Args:
        A_mod (ndarray): Mod. A matrix that has the Householder vectors
        tau (ndarray): Scaling factors for the Householder reflections
        
    Output:
        Q (ndarray): orthog Q
    """
    m, n = A_mod.shape
    Q = np.eye(m, n)  # This creates a m×n matrix with ones on the diagonal
    
    for j in range(n-1, -1, -1):
        # Extract the Householder vector
        w = np.zeros(m - j)
        w[0] = 1.0
        if j + 1 < m:
            w[1:] = A_mod[j+1:, j]
        
        # Apply the Householder reflection to Q
        wTQ = w @ Q[j:, :]
        update = np.outer(w, wTQ)
        Q[j:, :] -= tau[j] * update
    
    return Q

def extract_R(A_mod: np.ndarray) -> np.ndarray:
    """
    Extract the R matrix from the modified A matrix.
    
    Args:
        A_mod (ndarray): Modified A matrix containing the Householder vectors
    
    Output:
        R (ndarray): The upper triangular matrix R
    """
    m, n = A_mod.shape
    R = np.zeros((n, n))
    
    for i in range(n):
        R[i, i:] = A_mod[i, i:]
    
    return R

def QR_Householder(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute a reduced QR factorization for matrix A via 
    
    Args:
        A (ndarray): n x k, Input matrix with k <= n
    
    Output:
        Q (ndarray): n x k, orthog
        R (ndarray): k x k, upper triangular
    """
    A_mod, tau = QR_Householder_helper(A)
    Q = form_Q(A_mod, tau)
    R = extract_R(A_mod)
    return Q, R

In [ ]:
for k in range(1, 10):
  for n in range(k, 10):
    A = np.random.rand(5, 3)
    Q_hh, R_hh = QR_Householder(A)
    Q, R = np.linalg.qr(A, mode="reduced")
    assert(np.allclose(Q_hh @ R_hh, A))
    assert(np.allclose(Q_hh, Q))
    assert(np.allclose(R_hh, R))
print("All test cases passed")

All test cases passed


## Goal 2:
Using your $QR$ factorization, implement Algorithm 1. This is written as a matrix least squares problem, but think about the Frobenius norm and how you might be able to split it up into problems we know how to solve. **Please explain how you do this in the project report.** Practically, it is possible to then block some of these operations together for efficiency; you may take advantage of this if you wish.

In [53]:
# We will solve this by four steps
# 1. Backsolver for triangular
# 2. Matrix case
# 3. Implement Alternating LS

# https://www.cs.cornell.edu/courses/cs4220/2023sp/lec/2023-02-08.pdf
def backward_subst(U, d):
  """Backward substitution

  Args:
      U (ndarray):
      d (ndarray):

  Output:
      x (ndarray): 

  """
  n = d.shape[0]
  x = np.copy(d)
  for i in range(n-1, -1, -1):
      x[i] = (x[i] - np.sum(U[i, i+1:n] * x[i+1:n])) / U[i, i]
  return x

def matrix_ls(A, B):
  """Matrix Least Squares Solver via QR

  Args:
      A (ndarray): m x n, m >= n
      B (ndarray): m x p

  Output:
      X (ndarray): n x p, solution matrix
  """
  _, n = A.shape
  p = B.shape[1]
  X = np.zeros((n,p))

  Q, R = QR_Householder(A)

  for j in range(p):
      Qt_b = Q.T @ B[:, j]
      print(R, Qt_b)
      X[:, j] = backward_subst(R, Qt_b)
  
  return X

def alternating_ls(A, k, num_iterations):
  n1, n2 = A.shape

  W = np.random.randn(n1,k)
  Z = np.random.randn(n2,k)

  for _ in range(num_iterations):
    X = matrix_ls(W, A)
    Z = X.T

    X = matrix_ls(Z, A.T)
    W = X.T
  
  return W, Z

In [57]:
# Test backward_sub
for k in range(1, 10):
    for n in range(k, 10):
        U = np.triu(np.random.rand(n, n))
        for i in range(n):
            if abs(U[i, i]) < 1e-10:
                U[i, i] = 1.0
        d = np.random.rand(n)
        x_our = backward_subst(U, d)
        x_numpy = np.linalg.solve(U, d)
        assert np.allclose(x_our, x_numpy)
        assert np.allclose(U @ x_our, d)

print("All backward substitution tests passed!")

# Test matrix_ls
for i in range(5):
    rows = np.random.randint(5, 15)
    cols = np.random.randint(2, 5)
    b_cols = np.random.randint(1, 4) # Number of right-hand sides (columns in b)
    A = np.random.rand(rows, cols)
    B = np.random.rand(rows, b_cols)
    
    print(A, B)
    your_solution = matrix_ls(A, B)
    numpy_solution, _, _, _ = np.linalg.lstsq(A, B, rcond=None)
    
    # Compare the two solutions
    difference = np.linalg.norm(your_solution - numpy_solution)
    
    # Print results
    print(f"Test {i+1}:")
    print(f"  Matrix A shape: {A.shape}")
    print(f"  Matrix b shape: {b.shape}")
    print(f"  Solution difference: {difference:.10f}")
    
    # Simple pass/fail check
    if difference < 1e-10:
        print("  PASSED")
    else:
        print("  FAILED")
        print(f"  NumPy solution shape: {numpy_solution.shape}")
        print(f"  Your solution shape:  {your_solution.shape}")

# Test alternating_ls

All backward substitution tests passed!
[[0.34716684 0.5618981  0.43338085]
 [0.45891652 0.96957205 0.29615447]
 [0.62732447 0.42006267 0.06114618]
 [0.48933083 0.81334333 0.09875297]
 [0.06959185 0.15363152 0.76440011]
 [0.72399538 0.56501777 0.4682264 ]
 [0.88832581 0.76669782 0.81194225]
 [0.8132623  0.64271715 0.45801462]
 [0.86467775 0.92600742 0.02380604]
 [0.05155487 0.9587433  0.27716061]
 [0.87336944 0.38879512 0.11910006]
 [0.0804432  0.1565079  0.24989319]] [[0.48851835]
 [0.7014979 ]
 [0.98639858]
 [0.44150674]
 [0.51818494]
 [0.5218157 ]
 [0.53943436]
 [0.51774066]
 [0.00877476]
 [0.00172482]
 [0.09766632]
 [0.1015566 ]]
[1.16432911 1.39885095 1.23469869] [ 0.01356088 -2.02985564 -0.08250603]


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
import numpy as np

# Number of tests to run
num_tests = 5

# Simple for loop to test your matrix_ls function
for i in range(num_tests):
    # Generate random matrix dimensions
    rows = np.random.randint(5, 15)
    cols = np.random.randint(2, 5)
    
    # Create random matrix A and vector b
    A = np.random.rand(rows, cols)
    b = np.random.rand(rows)
    
    # Get solution from your function
    your_solution = matrix_ls(A, b)
    
    # Get solution from NumPy (the reference implementation)
    numpy_solution, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    
    # Compare the two solutions
    difference = np.linalg.norm(your_solution - numpy_solution)
    
    # Print results
    print(f"Test {i+1}:")
    print(f"  Matrix shape: {A.shape}")
    print(f"  Difference: {difference:.10f}")
    
    # Simple pass/fail check
    if difference < 1e-10:
        print("  PASSED")
    else:
        print("  FAILED")
        print(f"  NumPy solution: {numpy_solution}")
        print(f"  Your solution:  {your_solution}")
    
    print()

In [ ]:
import numpy as np

# Test backward_subst (already provided by your example)
def test_backward_subst():
    for k in range(1, 10):
        for n in range(k, 10):
            # Generate a random upper triangular matrix U of shape (n, n)
            U = np.triu(np.random.rand(n, n))
            # Ensure diagonal entries are not too small
            for i in range(n):
                if abs(U[i, i]) < 1e-10:
                    U[i, i] = 1.0
            d = np.random.rand(n)
            x_our = backward_subst(U, d)
            x_numpy = np.linalg.solve(U, d)
            assert np.allclose(x_our, x_numpy), f"Backward substitution failed for n={n}"
            assert np.allclose(U @ x_our, d), f"Solution does not satisfy Ux=d for n={n}"
    print("All backward substitution tests passed!")

test_backward_subst()



All backward substitution tests passed!


## Goal 3:
Using your implementation, load the file Cornell.csv which has an image stored in the matrix $C$ and try to compute the best rank 75 approximation of the image. Compare this with the result of the best rank 75 image from the SVD (you can use a built in routine to compute this), what do you observe qualitatively and quantitatively?

## Question 1:
Show that if we want to solve $$\min_x \|Ax-b\|_2^2 + \beta^2 \|x\|_2^2$$ we can instead solve 
$$
\min_x \left\|\begin{bmatrix}A \\ \beta I\end{bmatrix}x-\begin{bmatrix}b \\ 0\end{bmatrix}\right\|_2^2.
$$

## Goal 4:
Using your $QR$ factorization, implement Algorithm 2. From the first set of goals, you should have worked out how to split this up into solving many least squares problems. Now, you have to think about what size those problems are, and which entries of the matrices they involve. Given that, you can then leverage your $QR$ factorization and the preceding item to implement the algorithm. **Please explain how you do this in the project report.**

## Goal 5:
Using your implementation, load each of the image files in Image*.csv and associated masks Mask*.csv, where * is 1, 2, and 3. Each file contains an image and the associated mask file contains an image of the same size whose entries are 1 if the corresponding entry of $C$ is observed and 0 if it is unobserved (so it defines the set $\Omega$). The unknown entries of $C$ have been set arbitrarily and you should not access or use them (as the will adversely affect your results). Using Algorithm 2, try and recover each of the underlying images. You now have some parameters to consider and we encourage exploration of their values. You should not have to go to $k$ larger than 75 for any of the examples, and good $\beta$ to try are between $10^{-2}$ and 1. Report the images you recover, and discuss what you observe.